# TP Bioinformatique COVID-19 - Partie 3
## Contrôle qualité des séquences (FastQC)

**Auteur : Marwa zidi**

**Durée estimée : 20 minutes**

### Objectifs
- Utiliser FastQC pour évaluer la qualité des données de séquençage
- Interpréter les métriques de qualité
- Identifier les problèmes potentiels

---

In [ ]:
# Import des bibliothèques
import os
import subprocess
from Bio import SeqIO
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

print("✅ Bibliothèques chargées")

## 1. Introduction à FastQC

**FastQC** est l'outil standard pour le contrôle qualité des données de séquençage.

### Métriques analysées
1. **Per base sequence quality** : Qualité par position
2. **Per sequence quality scores** : Distribution des qualités moyennes
3. **Per base sequence content** : Composition en bases
4. **GC content** : Pourcentage GC
5. **Sequence length distribution** : Distribution des longueurs
6. **Duplicate sequences** : Taux de duplication
7. **Adapter content** : Contamination par adaptateurs

### Interprétation
- ✅ **PASS** : Analyse normale
- ⚠️ **WARN** : Résultats légèrement inhabituels
- ❌ **FAIL** : Problème potentiel

## 2. Analyse avec FastQC

Lançons FastQC sur nos données FASTQ.

In [ ]:
# Définition des chemins
fastq_file = "/opt/covid_data/sample_reads.fastq"
output_dir = "~/notebooks-covid/fastqc_results"

# Création du dossier de sortie
os.makedirs(os.path.expanduser(output_dir), exist_ok=True)

if os.path.exists(fastq_file):
    print("🔬 Lancement de FastQC...\n")
    
    # Commande FastQC
    cmd = f"fastqc {fastq_file} -o {os.path.expanduser(output_dir)} --quiet"
    
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    if result.returncode == 0:
        print("✅ FastQC terminé avec succès")
        print(f"📁 Résultats dans : {output_dir}")
        
        # Liste des fichiers générés
        files = os.listdir(os.path.expanduser(output_dir))
        print(f"\n📄 Fichiers générés :")
        for f in files:
            print(f"   - {f}")
    else:
        print(f"❌ Erreur : {result.stderr}")
else:
    print("⚠️ Fichier FASTQ non trouvé")
    print("\n💡 FastQC s'exécute normalement en ligne de commande :")
    print("   fastqc fichier.fastq -o output_dir/")

## 3. Analyse manuelle de la qualité (sans FastQC)

Créons nos propres graphiques de qualité similaires à FastQC.

In [ ]:
# Analyse de la qualité par position
if os.path.exists(fastq_file):
    # Lecture des reads
    reads = list(SeqIO.parse(fastq_file, "fastq"))[:10000]  # Premier 10k reads
    
    # Collecte des qualités par position
    max_length = max(len(read.seq) for read in reads)
    qualities_by_position = [[] for _ in range(max_length)]
    
    for read in reads:
        for pos, qual in enumerate(read.letter_annotations['phred_quality']):
            if pos < max_length:
                qualities_by_position[pos].append(qual)
    
    # Calcul des statistiques
    positions = list(range(1, max_length + 1))
    mean_quals = [np.mean(q) if q else 0 for q in qualities_by_position]
    q25_quals = [np.percentile(q, 25) if q else 0 for q in qualities_by_position]
    q75_quals = [np.percentile(q, 75) if q else 0 for q in qualities_by_position]
    
    # Visualisation
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
    
    # Graphique 1 : Qualité par position (style FastQC)
    ax1.fill_between(positions, q25_quals, q75_quals, alpha=0.3, color='yellow', label='Interquartile range')
    ax1.plot(positions, mean_quals, 'b-', linewidth=2, label='Mean quality')
    ax1.axhline(y=28, color='green', linestyle='--', linewidth=1, alpha=0.5)
    ax1.axhline(y=20, color='orange', linestyle='--', linewidth=1, alpha=0.5)
    ax1.axhspan(28, 40, alpha=0.1, color='green')
    ax1.axhspan(20, 28, alpha=0.1, color='yellow')
    ax1.axhspan(0, 20, alpha=0.1, color='red')
    
    ax1.set_xlabel('Position dans le read (pb)', fontsize=12)
    ax1.set_ylabel('Score de qualité Phred', fontsize=12)
    ax1.set_title('Qualité des bases par position (style FastQC)', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 42)
    
    # Graphique 2 : Distribution des qualités moyennes par read
    avg_qualities_per_read = [np.mean(read.letter_annotations['phred_quality']) for read in reads]
    ax2.hist(avg_qualities_per_read, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    ax2.axvline(x=30, color='red', linestyle='--', linewidth=2, label='Q30 threshold')
    ax2.set_xlabel('Qualité moyenne par read', fontsize=12)
    ax2.set_ylabel('Nombre de reads', fontsize=12)
    ax2.set_title('Distribution des qualités moyennes par séquence', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Statistiques
    print(f"\n📊 Statistiques de qualité ({len(reads)} reads analysés) :")
    print(f"   Qualité moyenne globale : Q{np.mean([q for pos in qualities_by_position for q in pos]):.1f}")
    print(f"   Reads avec qualité moyenne ≥ Q30 : {sum(1 for q in avg_qualities_per_read if q >= 30)/len(reads)*100:.1f}%")
    print(f"   Longueur moyenne des reads : {np.mean([len(r.seq) for r in reads]):.0f} pb")
else:
    print("⚠️ Pas de fichier FASTQ pour l'analyse")

## 4. Analyse du contenu en GC

Le pourcentage GC doit correspondre au génome attendu (SARS-CoV-2 : ~38%).

In [ ]:
# Calcul du GC% par read
if os.path.exists(fastq_file):
    reads = list(SeqIO.parse(fastq_file, "fastq"))[:5000]
    
    gc_contents = []
    for read in reads:
        gc_count = read.seq.count('G') + read.seq.count('C')
        gc_percent = (gc_count / len(read.seq)) * 100
        gc_contents.append(gc_percent)
    
    # Visualisation
    plt.figure(figsize=(12, 6))
    plt.hist(gc_contents, bins=50, edgecolor='black', alpha=0.7, color='teal')
    plt.axvline(x=38, color='red', linestyle='--', linewidth=2, 
                label='GC% théorique SARS-CoV-2 (38%)')
    plt.axvline(x=np.mean(gc_contents), color='orange', linestyle='-', linewidth=2, 
                label=f'GC% moyen observé ({np.mean(gc_contents):.1f}%)')
    plt.xlabel('GC content (%)', fontsize=12)
    plt.ylabel('Nombre de reads', fontsize=12)
    plt.title('Distribution du contenu en GC', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    print(f"\n🧬 Analyse du contenu GC :")
    print(f"   GC% moyen : {np.mean(gc_contents):.2f}%")
    print(f"   GC% attendu (SARS-CoV-2) : 38%")
    print(f"   Écart : {abs(np.mean(gc_contents) - 38):.2f}%")
    
    if abs(np.mean(gc_contents) - 38) < 3:
        print("   ✅ Contenu GC cohérent avec SARS-CoV-2")
    else:
        print("   ⚠️ Contenu GC différent de l'attendu (possible contamination)")
else:
    print("⚠️ Pas de fichier FASTQ pour l'analyse")

## 5. Détection de problèmes courants

### Problèmes typiques
1. **Baisse de qualité en fin de read** → Normal (artefact Illumina)
2. **GC% anormal** → Contamination possible
3. **Forte duplication** → Biais PCR ou couverture élevée
4. **Adaptateurs présents** → Nécessite trimming

### Critères d'acceptation
- ✅ **Q30 ≥ 75%** : Qualité acceptable
- ✅ **GC% proche de 38%** : Cohérent avec SARS-CoV-2
- ✅ **Pas d'adaptateurs** : Séquences propres

In [ ]:
# Résumé de la qualité
if os.path.exists(fastq_file):
    reads = list(SeqIO.parse(fastq_file, "fastq"))[:5000]
    
    # Calculs
    all_qualities = [q for read in reads for q in read.letter_annotations['phred_quality']]
    q30_percent = sum(1 for q in all_qualities if q >= 30) / len(all_qualities) * 100
    gc_avg = np.mean([(r.seq.count('G') + r.seq.count('C')) / len(r.seq) * 100 for r in reads])
    avg_length = np.mean([len(r.seq) for r in reads])
    
    print("\n" + "="*60)
    print("📋 RAPPORT DE QUALITÉ - RÉSUMÉ")
    print("="*60)
    print(f"\n📊 Métriques générales :")
    print(f"   Nombre de reads analysés : {len(reads):,}")
    print(f"   Longueur moyenne : {avg_length:.0f} pb")
    print(f"\n🎯 Qualité :")
    print(f"   Bases avec Q≥30 : {q30_percent:.2f}%", end="")
    if q30_percent >= 75:
        print(" ✅ Excellent")
    elif q30_percent >= 60:
        print(" ⚠️ Acceptable")
    else:
        print(" ❌ Problématique")
    
    print(f"\n🧬 Contenu GC :")
    print(f"   GC% moyen : {gc_avg:.2f}%", end="")
    if abs(gc_avg - 38) < 5:
        print(" ✅ Cohérent avec SARS-CoV-2")
    else:
        print(" ⚠️ Différent de l'attendu")
    
    print("\n" + "="*60)
    
    if q30_percent >= 75 and abs(gc_avg - 38) < 5:
        print("\n✅ DONNÉES DE BONNE QUALITÉ - Prêt pour l'alignement")
    else:
        print("\n⚠️ ATTENTION - Vérifier la qualité avant alignement")
    
    print("="*60)
else:
    print("⚠️ Pas de données pour le rapport")

## 📝 Points clés

✅ **FastQC** : Outil standard pour le contrôle qualité  
✅ **Q30** : Seuil minimum pour une bonne qualité (75% des bases)  
✅ **GC%** : Doit être cohérent avec l'organisme étudié  
✅ Baisse de qualité en fin de read = **normal** (Illumina)  
✅ Vérifier avant l'alignement pour éviter erreurs d'assemblage

---

**➡️ Passez au notebook suivant : `04_Alignement_Genome.ipynb`**